In [ ]:
import random
import os
import numpy as np
import time
import torch
import torch.nn as nn 
import pandas as pd 
import argparse  # 一个用于解析命令行参数的Python库。它允许你在命令行中指定程序的配置选项和参数，方便地控制程序的行为。
import torch.optim as optim # PyTorch中的优化器模块，包括各种优化算法，如SGD、Adam等。它用于优化神经网络的权重以最小化损失函数。
import torch.utils.data as data # PyTorch中的数据加载和处理工具，用于构建数据集和数据加载器，以便有效地加载和处理训练和测试数据。
from tensorboardX import SummaryWriter # 用于将日志数据写入TensorBoard日志文件，以便进行可视化和监控模型训练过程。


In [ ]:
ml_1m = pd.read_csv(
    './mooc_data/filtered.csv', 
    sep=",", 
    names = ['user_id', 'item_id', 'rating', 'timestamp'], 
    engine='python')

In [ ]:
def seed_everything(seed):  #设置伪随机数，方便实验结果复现
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 即使没有 CUDA，也可以设置，不会产生影响
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # 在没有 CUDA 的情况下通常将其设置为 False

In [ ]:
import sys
sys.argv = ['run.py']
parser = argparse.ArgumentParser() 
parser.add_argument("--seed", type=int, default=42,help="Seed")
parser.add_argument("--lr", type=float, default=0.001, help="learning rate")
parser.add_argument("--dropout",type=float,default=0.2, help="dropout rate")
parser.add_argument("--batch_size", type=int, default=256,help="batch size for training")
parser.add_argument("--epochs", type=int,default=3,  help="training epoches")
parser.add_argument("--top_k", type=int, default=10, help="compute metrics@top_k")  # 用于计算评估指标的top-k值，通常用于评估模型的性能。默认值为10。
parser.add_argument("--factor_num", type=int,default=32, help="predictive factors numbers in the model")  # 模型中的潜在因子数量，通常用于嵌入层的维度。默认值为32
parser.add_argument("--layers",nargs='+', default=[64,32,16,8], help="MLP layers. Note that the first layer is the concatenation of user and item embeddings. So layers[0]/2 is the embedding size.")  # MLP（多层感知器）的层次结构，即神经网络中的隐藏层。默认设置为[64, 32, 16, 8]，表示模型包含4个隐藏层，每层的神经元数量分别是64、32、16和8。
parser.add_argument("--num_ng", type=int,default=4, help="Number of negative samples for training set")  # 用于训练集的负样本数量，即每个正样本对应的负样本数量。默认值为4。
parser.add_argument("--num_ng_test", type=int,default=100, help="Number of negative samples for test set")  # 用于测试集的负样本数量，即每个正样本对应的负样本数量。默认值为100。
parser.add_argument("--out", default=True,help="save model or not")  

# set device and parameters
args=parser.parse_known_args()[0]  # args = parser.parse_args()有问题解决看： https://blog.csdn.net/wmq104/article/details/123534597
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter()  #将各种信息写入 TensorBoard 日志文件，以便在 TensorBoard 界面中查看和分析实验结果。

seed_everything(args.seed)

数据载入：

In [ ]:
"""
这个自定义数据集类的主要目的是将原始数据组织成适用于 PyTorch 模型训练和测试的形式。
通过实现这些方法，可以方便地使用 PyTorch 提供的数据加载器来加载和处理数据，以供深度学习模型使用。
"""
class Rating_Datset(torch.utils.data.Dataset):
    def __init__(self, user_list, item_list, rating_list):
        super(Rating_Datset, self).__init__()
        self.user_list = user_list
        self.item_list = item_list
        self.rating_list = rating_list

    def __len__(self):
        return len(self.user_list)

    def __getitem__(self, idx):
        user = self.user_list[idx]
        item = self.item_list[idx]
        rating = self.rating_list[idx]

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(item, dtype=torch.long),
            torch.tensor(rating, dtype=torch.float)
            )

In [ ]:
class MLP_Data(object):
    """
    Construct Dataset for MLP
    """
    def __init__(self, args, ratings):
        self.ratings = ratings
        self.num_ng = args.num_ng
        self.num_ng_test = args.num_ng_test
        self.batch_size = args.batch_size

        self.preprocess_ratings = self._reindex(self.ratings)

        self.user_pool = set(self.ratings['user_id'].unique())
        self.item_pool = set(self.ratings['item_id'].unique())

        self.train_ratings, self.test_ratings = self._leave_one_out(self.preprocess_ratings)
        self.negatives = self._negative_sampling(self.preprocess_ratings)
        random.seed(args.seed)

        
    """
    本函数作用：
    1.用整数重新索引
    2.将有无交互映射为0,1
    """   
    def _reindex(self, ratings):
        """
        Process dataset to reindex userID and itemID, also set rating as binary feedback
        """
        user_list = list(ratings['user_id'].drop_duplicates())
        user2id = {w: i for i, w in enumerate(user_list)}
        item_list = list(ratings['item_id'].drop_duplicates())
        item2id = {w: i for i, w in enumerate(item_list)}  #  enumerate(item_list) 是一个迭代器，用于遍历 item_list 列表中的元素。在每次迭代中，enumerate 返回一个元组，其中第一个元素 i 是元素的索引（从0开始），第二个元素 w 是元素的值（即物品ID）。
        ratings['user_id'] = ratings['user_id'].apply(lambda x: user2id[x])
        ratings['item_id'] = ratings['item_id'].apply(lambda x: item2id[x])  # .apply() 方法的作用，ratings['item_id'] 列中的每个原始物品ID都被替换为了相应的整数ID，完成了物品ID的重新索引
        ratings['rating'] = ratings['rating'].apply(lambda x: float(x > 0))  # .apply() 方法的作用，ratings['rating'] 列中的每个原始评分值都被替换为了相应的二进制反馈值。
        return ratings
                                             
                                                
    """
    这种 "leave-one-out" 评估协议的核心思想是将每个用户的最新交互数据作为测试集，其余数据作为训练集，以评估模型的性能。
    这有助于模拟实际应用场景中的推荐任务，其中用户的历史行为用于训练模型，最新行为用于测试模型。
    """
    def _leave_one_out(self, ratings):
        """
        leave-one-out evaluation protocol in paper https://www.comp.nus.edu.sg/~xiangnan/papers/ncf.pdf
        """
        ratings['rank_latest'] = ratings.groupby(['user_id'])['timestamp'].rank(method='first', ascending=False)
        test = ratings.loc[ratings['rank_latest'] == 1]
        train = ratings.loc[ratings['rank_latest'] > 1]
        assert train['user_id'].nunique()==test['user_id'].nunique(), 'Not Match Train User with Test User'
        return train[['user_id', 'item_id', 'rating']], test[['user_id', 'item_id', 'rating']]


    def _negative_sampling(self, ratings):
        interact_status = (#一个DataFrame
            ratings.groupby('user_id')['item_id']
            .apply(set)  #.apply(set)：然后，对于每个用户，它将其交互过的物品ID放入一个集合（set）中，以去除重复的物品。
            .reset_index()  #它重置索引，以便得到一个包含用户ID和其交互过的物品ID集合的 DataFrame。
            .rename(columns={'item_id': 'interacted_items'}))
        interact_status['negative_items'] = interact_status['interacted_items'].apply(lambda x: self.item_pool - x)  #通过从整体物品池 self.item_pool 中减去每个用户的交互物品集合得到的。这意味着 negative_items 包含了每个用户未曾交互的物品。
        interact_status['negative_samples'] = interact_status['negative_items'].apply(lambda x: random.sample(x, self.num_ng_test))  #negative_samples 的列，其中包含了每个用户的负样本样本。这是通过从 negative_items 中随机抽样 self.num_ng_test 个物品得到的
        return interact_status[['user_id', 'negative_items', 'negative_samples']]

    def get_train_instance(self):
        users, items, ratings = [], [], []
        train_ratings = pd.merge(self.train_ratings, self.negatives[['user_id', 'negative_items']], on='user_id')  # 将训练评分数据集 self.train_ratings 与负样本数据 self.negatives[['user_id', 'negative_items']] 进行合并，以获得包含正样本和负样本的数据                                              
        train_ratings['negatives'] = train_ratings['negative_items'].apply(lambda x: random.sample(x, self.num_ng))  # 创建了一个新的列 negatives，其中包含了每个用户的负样本物品列表，这些负样本物品是从 negative_items 中随机抽样的，数量由 self.num_ng 决定                                              
        for row in train_ratings.itertuples():
            users.append(int(row.user_id))
            items.append(int(row.item_id))
            ratings.append(float(row.rating))
            for i in range(self.num_ng):  #添加负样本
                users.append(int(row.user_id))
                items.append(int(row.negatives[i]))
                ratings.append(float(0))  # negative samples get 0 rating
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)#num_workers=4改成0，4会发生broken pipe

    def get_test_instance(self):
        users, items, ratings = [], [], []
        test_ratings = pd.merge(self.test_ratings, self.negatives[['user_id', 'negative_samples']], on='user_id')
        for row in test_ratings.itertuples():
            users.append(int(row.user_id))
            items.append(int(row.item_id))
            ratings.append(float(row.rating))
            for i in getattr(row, 'negative_samples'):
                users.append(int(row.user_id))
                items.append(int(i))
                ratings.append(float(0))
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.num_ng_test+1, shuffle=False, num_workers=0)#num_workers=4改成0


In [ ]:
# construct the train and test datasets
data = MLP_Data(args, ml_1m)
train_loader =data.get_train_instance()
test_loader =data.get_test_instance()

MLP：

In [ ]:
class Multi_Layer_Perceptron(nn.Module):
    def __init__(self, args, num_users, num_items):
        super(Multi_Layer_Perceptron, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.factor_num = args.factor_num
        self.layers = args.layers

        self.embedding_user = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num)
        self.embedding_item = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num)

        self.fc_layers = nn.ModuleList()#创建了空列表用于存储多个全连接层
        for idx, (in_size, out_size) in enumerate(zip(self.layers[:-1], self.layers[1:])):#切片的语法
# 这是一个循环，它同时迭代遍历 self.layers[:-1] 和 self.layers[1:] 中的元素对，将它们分配给变量 idx（索引）、in_size（输入维度）和 out_size（输出维度）。
            self.fc_layers.append(nn.Linear(in_size, out_size))
# append 方法将这个全连接层对象添加到 self.fc_layers 中，将其存储起来以备后续使用
        self.affine_output = nn.Linear(in_features=self.layers[-1], out_features=1)
        self.logistic = nn.Sigmoid()

    def forward(self, user_indices, item_indices):
        user_embedding = self.embedding_user(user_indices)
        item_embedding = self.embedding_item(item_indices)
        vector = torch.cat([user_embedding, item_embedding], dim=-1)  # the concat latent vector
        for idx, _ in enumerate(range(len(self.fc_layers))):
            vector = self.fc_layers[idx](vector)
            vector = nn.ReLU()(vector)
            # vector = nn.BatchNorm1d()(vector)
            # vector = nn.Dropout(p=0.5)(vector)
        logits = self.affine_output(vector)
        rating = self.logistic(logits)
        return rating.squeeze()

    def init_weight(self):
        pass

损失函数：

In [ ]:
loss_function = nn.BCELoss()

评估：

In [ ]:
def hit(ng_item, pred_items):
    if ng_item in pred_items:
        return 1
    return 0

def ndcg(ng_item, pred_items):
    if ng_item in pred_items:
        index = pred_items.index(ng_item)
        return np.reciprocal(np.log2(index+2))
    return 0

def metrics(model, test_loader, top_k, device):
    HR, NDCG = [], []

    for user, item, label in test_loader:
        user = user.to(device)
        item = item.to(device)

        predictions = model(user, item)
        _, indices = torch.topk(predictions, top_k)
        recommends = torch.take(
                item, indices).cpu().numpy().tolist()

        ng_item = item[0].item() # leave one-out evaluation has only one item per user
        HR.append(hit(ng_item, recommends))
        NDCG.append(ndcg(ng_item, recommends))

    return np.mean(HR), np.mean(NDCG)

训练：

In [ ]:
MODEL_PATH = r'/JupyterCode/Data_balanced'
MODEL = './models/mlp.pth'

# set the num_users, items
num_users = ml_1m['user_id'].nunique()+1  # ml_1m['user_id'].nunique()：这部分代码首先从 ml_1m 数据集中选择列 user_id，然后使用 nunique() 方法来计算唯一用户的数量。每个用户通常有一个唯一的用户ID，所以这个操作返回数据集中不同用户的数量。
num_items = ml_1m['item_id'].nunique()+1  # +1：通常在计算用户数量和物品数量时，会将一个额外的数值加上去，以便考虑到可能的用户或物品ID从0开始的情况。

# set model and optimizer
model =Multi_Layer_Perceptron(args, num_users, num_items)
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=args.lr)

# train, evaluation
best_hr = 0
for epoch in range(1, args.epochs+1):
    model.train() 
        # Enable dropout (if have).
        # model.train() 是 PyTorch 中的一个方法，用于将模型设置为训练模式。让我解释一下这个方法的作用和用途：
        # 在深度学习中，训练模型和评估模型是两个不同的模式。模型在训练模式下时，它会执行以下操作：
        # 启用 dropout 和批量归一化（如果模型中使用了这些操作）。这意味着在训练模式下，模型会在前向传播过程中应用 dropout 层，
        # 以防止过拟合，并且会更新批量归一化的统计信息。
        # 模型会保留每个中间层的梯度，以便进行反向传播和参数更新。
        # 在训练模式下，通常会使用不同的数据批次进行多次迭代，以便更新模型参数来最小化损失函数。
    all_timel_start = time.time()
    for user, item, label in train_loader:
        start_time = time.time()
        user = user.to(device)
        item = item.to(device)
        label = label.to(device)

        optimizer.zero_grad()  #清零梯度
        prediction = model(user, item)
            # 在 PyTorch 中，当你定义了一个继承自 nn.Module 的模型类时，模型的前向传播逻辑会被实现在 forward 方法中。
            # 当你调用模型实例（例如，model(user, item)）时，PyTorch 会自动调用模型的 forward 方法来执行前向传播操作，
            # 因此你不需要显式调用 model.forward()。
            # 简而言之，model(user, item) 与 model.forward(user, item) 是等效的。在代码中，一般会使用 model(user, item) 来执行前向传播，
            # 因为这更简洁和方便。 PyTorch 的设计让模型的使用更接近自然语言，提高了代码的可读性。
        
        loss = loss_function(prediction, label)  # 度量模型的预测与真实标签之间的差异
        loss.backward()  # 反向传播的过程，它用于计算如何更新模型的参数以降低损失
        optimizer.step()  # 优化器根据计算得到的梯度来更新模型的权重和偏置，以降低损失函数的值
        writer.add_scalar('loss/Train_loss', loss.item(), epoch)  # 将训练损失值记录到 TensorBoard 日志中，以便后续可视化和分析。这有助于监控模型的训练过程，了解损失如何随着时间（epoch）的推移而变化。
    
    model.eval()
        #     model.eval() 是 PyTorch 中的一个方法，用于将神经网络模型切换到评估（inference）模式。
        # 当调用这个方法时，模型会进入一个状态，该状态适用于在验证集或测试集上进行推断操作，而不是训练。
        # 主要的变化包括以下几点：
        # Dropout 和 Batch Normalization 的影响： 在训练时，神经网络通常使用 dropout 层和批归一化（Batch Normalization）层来防止过拟合和加速训练。但在评估模型性能时，我们不希望随机丢弃神经元或改变批次统计信息。因此，model.eval() 会将 dropout 层和批归一化层的行为设置为不使用（即固定参数），以确保模型在评估时的输出是确定性的。
        # 梯度计算： 在评估模式下，PyTorch 不会计算梯度信息，因为我们不需要在推断时进行参数更新。这节省了计算资源，并且可以提高模型的计算速度。
            
    HR, NDCG = metrics(model, test_loader, args.top_k, device)
    writer.add_scalar('Perfomance/HR@10', HR, epoch)
    writer.add_scalar('Perfomance/NDCG@10', NDCG, epoch)

    elapsed_time = time.time() - start_time
    print("The time elapse of epoch {:03d}".format(epoch) + " is: " + 
            time.strftime("%H: %M: %S", time.gmtime(elapsed_time)))
    print("HR: {:.5f}\tNDCG: {:.5f}".format(np.mean(HR), np.mean(NDCG)))

    if HR > best_hr:
        all_time = time.time() - all_timel_start
        best_hr, best_ndcg, best_epoch = HR, NDCG, epoch
        if args.out:
            if not os.path.exists(MODEL_PATH):#删去了config.
                os.mkdir(MODEL_PATH)
            torch.save(model.state_dict(), 
                '{}'.format(MODEL))

writer.close()
print("End. Best epoch {:03d}: HR = {:.5f}, NDCG = {:.5f}".format(best_epoch, best_hr, best_ndcg))

The time elapse of epoch 001 is: 00: 01: 07
HR: 0.62444	NDCG: 0.35530
The time elapse of epoch 002 is: 00: 01: 03
HR: 0.63202	NDCG: 0.36686
The time elapse of epoch 003 is: 00: 01: 01
HR: 0.63613	NDCG: 0.39983
End. Best epoch 003: HR = 0.63613, NDCG = 0.39983


In [ ]:

import torch.nn.functional as F
from numpy.linalg import norm
from sklearn.cluster import KMeans
from collections import defaultdict

model.eval()

#计算相似度
def cosine_similarity(u,v):
    return np.dot(u,v) / (norm(u)*norm(v))

user_embeddings = model.embedding_user.weight.detach().cpu().numpy()



In [ ]:
from Bio.Cluster import kcluster
from Bio.Cluster import clustercentroids

# 设置聚类的参数
n=15
num_clusters = n
method = 'u'  # 使用 'u' 方法进行聚类

# 调用 kcluster 函数进行聚类
clusterid, error, nfound = kcluster(user_embeddings, nclusters=num_clusters, dist=method,npass=100)

# 打印聚类结果
print("Cluster IDs:", clusterid)

# 计算聚类中心的嵌入
centroids, _ = clustercentroids(user_embeddings, clusterid=clusterid)

# 打印聚类中心的嵌入
print("Cluster centroids:\n", centroids)


Cluster IDs: [12  4  6 ...  0 13  9]
Cluster centroids:
 [[-5.65132158e-01  2.47855045e-01  6.73772311e-01 -6.61432246e-01
  -9.11238647e-02 -1.14900827e-01  1.66178896e-03  6.48082087e-02
   1.69916287e-02  3.12086064e-01 -3.07655290e-01  5.24310066e-01
  -1.61648811e-01 -9.75695948e-03  4.11068252e-01 -4.15245259e-01
  -5.89273544e-01 -4.46438181e-01 -1.74201400e-01 -2.40550770e-01
  -5.76891978e-01  1.64348634e-01 -5.94536244e-01  2.41892582e-01
  -4.37387461e-01 -2.77113999e-01 -8.84059982e-02 -1.65698256e-01
   2.74044188e-01 -2.03997862e-01 -1.00401588e-01  2.21191481e-01]
 [ 2.21841438e-01  7.45699357e-01  1.04169214e-01 -3.45217035e-01
  -3.48561236e-01  2.20872471e-01 -8.57658276e-02 -1.62745913e-01
  -2.45320969e-01 -3.95279053e-01  1.40596795e-01 -2.06971954e-01
   2.76237879e-01  3.33318543e-01  2.00085805e-01 -2.77505036e-02
   1.51313543e-01 -6.41733047e-01  2.84967002e-01 -1.99683893e-01
  -6.91003374e-02  1.72640705e-01  3.18968714e-01  2.51388678e-01
   6.02087668e-01 

In [ ]:
def find_available_subset(subsets, subset_samples_nums, centroids, user_embeddings, user_id, t, n, current_index):
    for k in range(n):
        if k != current_index and subset_samples_nums[k] < t:
            return k
    # 如果当前聚类中心对应的子集都已经达到了大小限制t，则递归地在剩余的聚类中心中寻找
    for i in range(n):
        if i != current_index:
            for j in range(n):
                if i != j:
                    c[j] = cosine_similarity(centroids[j], user_embeddings[user_id])
            max_index = c.index(max(c))
            if subset_samples_nums[max_index] >= t:
                # 继续递归寻找
                return find_available_subset(subsets, subset_samples_nums, centroids, user_embeddings, user_id, t, n, max_index)
            else:
                return max_index

In [ ]:
import pandas as pd

# 加载原始训练数据
train_data = pd.read_csv('./data_div_train_and_test/train_data.csv',names=['user_id', 'item_id', 'interaction'])
train_data_samples_num = train_data.shape[0]

t=train_data_samples_num/n

user_groups = train_data.groupby('user_id')

subsets = [pd.DataFrame() for _ in range(n)]
subset_samples_nums = [subset.shape[0] for subset in subsets]

c = [0] * n  # 初始化c为长度为n的零数组

for user_id, group in user_groups:
    cluster_label = clusterid[user_id]  # 假设 cluster_labels 是一个字典，键是用户ID，值是聚类标签
    flag=False
    for i in range(n):
        if cluster_label == i and subset_samples_nums[i] < t:
            subsets[i] = pd.concat([subsets[i], group])
            subset_samples_nums[i] = subsets[i].shape[0]
            flag = True
        elif cluster_label == i and subset_samples_nums[i] >= t:
            for j in range(n):
                if i != j:
                    c[j] = cosine_similarity(centroids[j], user_embeddings[user_id])
            max_index = c.index(max(c))
            if subset_samples_nums[max_index] < t:
                subsets[max_index] = pd.concat([subsets[max_index], group])
                subset_samples_nums[max_index] = subsets[max_index].shape[0]
                flag = True
            elif subset_samples_nums[max_index] >= t:
                # 递归寻找小于t的子集
                index_to_add = find_available_subset(subsets, subset_samples_nums, centroids, user_embeddings, user_id, t, n, i)
                subsets[index_to_add] = pd.concat([subsets[index_to_add], group])
                subset_samples_nums[index_to_add] = subsets[index_to_add].shape[0]
                flag = True
        if flag:
            break
            
                
# 可以选择保存这些子集到文件中
for r in range(n):
    subsets[r].to_csv(f'./data_div/div{n}/sub_train{r+1}.csv', index=False,header=False)




In [ ]:
import winsound
duration = 1000  # millisecond
freq = 440  # Hz
winsound.Beep(freq, duration)